In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

!pip install -q bitsandbytes transformers accelerate peft


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel


BASE_MODEL_ID = "tarun7r/Finance-Llama-8B"
LORA_PATH = "/content/drive/MyDrive/finance_lora_checkpoints_v3"  # change if needed

# 4-bit quantization (QLoRA-style) to save memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (general finance)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()

print("Loading LoRA adapter (risk head)...")
risk_model = PeftModel.from_pretrained(base_model, LORA_PATH)
risk_model.eval()

print("Models ready.")


In [ ]:
INSTRUCTION = (
    "You are a financial risk analysis model.\n"
    "Read the company text and generate a concise, structured risk report "
    "using EXACTLY the following fields and format, one field per line:\n\n"
    "risk_severity: \n"
    "risk_categories: \n"
    "financial_impact: \n"
    "key_metrics: \n"
    "critical_dates: \n"
    "analysis: <2–4 sentence narrative>\n\n"
    "Rules:\n"
    "- ALWAYS include every field.\n"
    "- Output MUST be plain text only (no JSON, no braces).\n"
    "- No quotes, no bullets, no extra commentary.\n"
    "- Write clearly and professionally."
)

def build_risk_prompt(text: str) -> str:
    return (
        INSTRUCTION
        + "\n\nCompany text:\n"
        + text
        + "\n\nRisk report:\n"
    )


In [ ]:
def clean_financial_impact_block(report: str) -> str:
    """
    Quick win: do NOT try to normalize to pretty numbers.
    Just keep whatever model said; if it hallucinated JSON, strip braces.
    """
    lines = report.splitlines()
    new_lines = []
    for line in lines:
        lower = line.lower().strip()
        if lower.startswith("financial_impact:"):
            # Remove any {...} blobs just to avoid ugly JSON in UI
            raw = line.split(":", 1)[1].strip()
            # Strip leading/trailing braces if present
            raw = raw.strip()
            if raw.startswith("{") and raw.endswith("}"):
                # Replace with a generic sentence
                cleaned = (
                    "Financial impact mentioned qualitatively; "
                    "exact amount and timing may not be precise."
                )
            elif not raw:
                cleaned = (
                    "Financial impact discussed qualitatively; no clear quantified "
                    "loss or gain disclosed."
                )
            else:
                cleaned = raw
            new_lines.append("financial_impact: " + cleaned)
        else:
            new_lines.append(line)
    return "\n".join(new_lines)


def generate_risk_report(company_text: str,
                         max_new_tokens: int = 600,
                         temperature: float = 0.3,
                         top_p: float = 0.95) -> str:
    """
    Uses the LoRA risk head to generate the 6-line risk report.
    """
    prompt = build_risk_prompt(company_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(risk_model.device)

    with torch.no_grad():
        gen = risk_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.05,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    raw = tokenizer.decode(gen[0], skip_special_tokens=True)

    if "Risk report:" in raw:
        out = raw.split("Risk report:")[-1].strip()
    else:
        out = raw.strip()

    # drop empty lines
    out = "\n".join([l.rstrip() for l in out.splitlines() if l.strip()])
    # generic cleaning of financial_impact
    out = clean_financial_impact_block(out)
    return out


In [ ]:
def generate_base_answer(question: str,
                         context: str = None,
                         max_new_tokens: int = 220,
                         temperature: float = 0.3,
                         top_p: float = 0.9) -> str:
    """
    General finance answer generator.

    - Uses slightly higher max_new_tokens to avoid truncation.
    - Asks the model explicitly to finish its explanation.
    """

    if context:
        prompt = (
            "You are a highly knowledgeable finance assistant.\n"
            "Use the CONTEXT to answer the QUESTION accurately, but do not copy text verbatim.\n"
            "You must produce only a final narrative answer for a retail investor.\n"
            "Do NOT define any output format, fields, or templates.\n"
            "Do NOT say that you will provide a structured report.\n"
            "Write one short, complete paragraph of 4–6 sentences of plain text only.\n"
            "Make sure you finish your explanation and do not stop in the middle of a sentence.\n\n"
            "=== CONTEXT ===\n"
            f"{context}\n\n"
            "=== QUESTION ===\n"
            f"{question}\n\n"
            "=== FINAL ANSWER ===\n"
        )
    else:
        prompt = (
            "You are a highly knowledgeable finance assistant.\n"
            "Answer the question as a clear explanation for a retail investor.\n"
            "Do NOT define any output formats, fields, or templates.\n"
            "Do NOT say that you will provide a structured report.\n"
            "Write one short, complete paragraph of 4–6 sentences of plain text only.\n"
            "Make sure you finish your explanation and do not stop in the middle of a sentence.\n\n"
            "Question:\n"
            f"{question}\n\n"
            "Final answer:\n"
        )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(base_model.device)

    with torch.no_grad():
        gen_ids = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    full = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

    # Strip prompt
    if "=== FINAL ANSWER ===" in full:
        answer = full.split("=== FINAL ANSWER ===", 1)[-1].strip()
    elif "Final answer:" in full:
        answer = full.split("Final answer:", 1)[-1].strip()
    else:
        if full.startswith(prompt):
            answer = full[len(prompt):].strip()
        else:
            answer = full.strip()

    # Cut off any residual risk-template junk if it ever appears
    cut_markers = [
        "Read the company text and generate a concise, structured",
        "using EXACTLY the following fields",
        "risk_severity:",
        "financial_impact:",
        "Risk report:",
    ]
    for m in cut_markers:
        idx = answer.find(m)
        if idx != -1:
            answer = answer[:idx].rstrip()
            break

    return answer

In [ ]:
def is_risk_intent(question: str) -> bool:
    """
    Cheap heuristic: look for risk-related words.
    """
    if not question:
        return True
    q = question.lower()
    triggers = [
        "risk", "market risk", "liquidity", "debt", "leverage",
        "covenant", "fx", "foreign currency", "interest rate",
        "hedge", "hedging"
    ]
    return any(word in q for word in triggers)

def build_risk_commentary_prompt(company_text: str, risk_report: str) -> str:
    return (
        "You are a senior financial risk officer.\n"
        "You are given:\n"
        "1) Company text from a filing or disclosure.\n"
        "2) A structured risk report produced by an internal model.\n\n"
        "Write a short, professional explanation (one paragraph, 3–5 sentences) that:\n"
        "- Explains the main market, interest-rate, debt and liquidity risks.\n"
        "- Uses figures from the original company text when available.\n"
        "- If any number in the report conflicts with the text, rely on the text.\n"
        "- Do NOT repeat the six field names.\n"
        "- Do NOT rewrite the risk report as a bullet list.\n"
        "- Focus on interpretation and implications for the firm's risk profile.\n\n"
        "=== COMPANY TEXT ===\n"
        f"{company_text}\n\n"
        "=== RISK REPORT ===\n"
        f"{risk_report}\n\n"
        "=== EXPLANATION ===\n"
    )

def finance_router(user_question: str, company_text: str = ""):
    """
    - If company_text given and intent is risk-related -> 6-line risk report + commentary.
    - Else -> general finance answer from base model.
    """
    company_text = company_text.strip()

    # Risk path
    if company_text and is_risk_intent(user_question):
        risk_report = generate_risk_report(company_text)
        commentary_prompt = build_risk_commentary_prompt(company_text, risk_report)
        commentary = generate_base_answer(
            question="Explain the firm's risk profile based on the above.",
            context=commentary_prompt,
            max_new_tokens=512,
        )
        combined = risk_report.strip() + "\n\n" + commentary.strip()
        return {"answer": combined}

    # General finance path
    answer = generate_base_answer(
        question=user_question or "Explain the following financial text.",
        context=company_text or None,
        max_new_tokens=1024,
    )
    return {"answer": answer}


In [ ]:

user_question = """You are 20 years old with a monthly salary of $5,000. After living expenses, EMIs, and shopping, you have $3,000 per month available for investing. Your employer gives you a 5% salary increase every year, and you plan to increase the percentage of your income that you invest by approximately 3% each year.

Design a comprehensive financial plan that answers the following:

How should you allocate your initial $3,000 monthly investable surplus across the following categories:
• Stocks (individual equities or equity funds)
• Corporate bonds
• Real estate exposure (e.g., REITs or saving toward property)
• Gold and silver investments
• Insurance (life, health, and any other critical risk protections)

For each category, specify the percentage and dollar amount you would invest initially and explain why that allocation is appropriate for a 20-year-old with a long investment horizon and moderate risk tolerance based on goals like wealth growth, risk protection, and diversification.

Explain how you would adjust these allocations over time as your salary rises with annual 5% hikes and as you increase your investment rate by ~3% each year. Include how you would rebalance your portfolio over the next 10–15 years to reflect changing goals (e.g., accumulating a home down payment, retirement planning).

Describe the role of retirement planning in your strategy, including what retirement vehicles you would use and how your long-term retirement allocation might evolve over time (e.g., shifting from more aggressive equities to more conservative bonds/other assets as you age).

Explain what types of insurance you need at your age and how they fit into your overall financial plan, including how much coverage you should aim for relative to your income and dependents.

Discuss how commodities like gold and silver fit into your diversified portfolio, including what percentage allocation is appropriate and the rationale behind including them relative to stocks, bonds, and real estate"""
company_text = """"""

result = finance_router(user_question=user_question, company_text=company_text)

# print("MODE:", result["mode"])
print("\n=== ANSWER ===")
print(result["answer"])
